## DataSet

In [10]:
from pydantic import BaseModel, Field
from pathlib import Path
from ids_expt.core.defs import DataType


class PCAPImageDataConfig(BaseModel):
    data_root: Path = Field(
        default=Path(r"E:\MSc Works\IDS\notebooks\output"),
        description="Root directory for storing PCAP images.",
    )
    train_ratio: float = Field(
        default=0.75,
        ge=0.0,
        le=1.0,
        description="Proportion of data to use for training.",
    )
    random_seed: int = Field(
        default=42,
        description="Random seed for reproducibility.",
    )
    labels: list[str] = Field(
        default=[],
        description="List of labels to be used for classification.",
    )
    normal_label: str = Field(
        default="NORMAL",
        description="Label for normal data samples.",
    )
    combine_attacks: bool = Field(
        default=False,
        description="Whether to combine all attack types into a single label.",
    )
    max_data: int = Field(
        default=-1,
        description="Maximum number of samples to be used from each class.",
    )
    byte_length: int = Field(
        default=8 * 32,
        ge=1,
        description="Length of byte sequences to be used for image generation.",
    )
    num_pkts: int = Field(
        default=6 * 32,
        ge=1,
        description="Number of packets to consider for each sample.",
    )

In [60]:
from loguru import logger
import pandas as pd
from sklearn.model_selection import train_test_split
import json
import pickle
from ids_expt.core.defs import Session
from scapy.all import raw
from tqdm import tqdm
import cv2
from torch.utils.data import Dataset as TorchDataset
import torch


def image_normalize(image):
    """Normalize image pixel values to [0, 1] range."""
    return image / 255.0


class DFDataSet:
    def __init__(self, config: PCAPImageDataConfig):
        self.config = config
        self.data_root = config.data_root
        self.train_ratio = config.train_ratio
        self.data_df = None
        self.data_type = None
        self.label_encoding = None
        self.scaler = image_normalize

    def read_statistics(self):
        self.all_sessions = list(self.data_root.rglob("*.pkl"))
        logger.info(f"Found {len(self.all_sessions)} session files in {self.data_root}")
        stats = []
        session_index = 0
        for session_file in tqdm(self.all_sessions):
            # session_file = self.all_sessions[43]
            with open(session_file, "rb") as f:
                try:
                    sessions: list[Session] = pickle.load(f)
                except Exception as e:
                    logger.error(f"Error reading {session_file}: {e}")
                    continue
            for sidx, session in enumerate(sessions):
                session_pkts = session.packets
                num_pkts = len(session_pkts)
                label = session.label
                duration = session.duration
                pkt_lens = []
                for pkt in session_pkts:
                    pkt_raw = raw(pkt)
                    pkt_lens.append(len(pkt_raw))
                min_len = min(pkt_lens) if pkt_lens else 0
                max_len = max(pkt_lens) if pkt_lens else 0
                stats.append(
                    {
                        "session_file": session_file.name,
                        "session_index": session_index,
                        "num_packets": num_pkts,
                        "label": label,
                        "duration": duration,
                        "min_packet_length": min_len,
                        "max_packet_length": max_len,
                    }
                )
                session_index += 1
        self.stats_df = pd.DataFrame(stats)
        return self.stats_df

    def load_data(self):
        self.all_files = list(self.data_root.rglob("*.png"))
        logger.info(f"Found {len(self.all_files)} files in {self.data_root}")
        self.all_files = [
            f for f in self.all_files if f.is_file() and "hot" not in f.stem
        ]
        logger.info(
            f"Filtered to {len(self.all_files)} files after removing 'hot' images"
        )
        data = []
        for file in self.all_files:
            label = file.stem.replace("session_", "").split("_")[1:]
            label = "_".join(label)
            data.append({"file_path": str(file), "label": label})
        self.data_df = pd.DataFrame(data)
        if self.config.combine_attacks:
            self.data_df["label"] = self.data_df["label"].apply(
                lambda x: (
                    self.config.normal_label
                    if x == self.config.normal_label
                    else "ATTACK"
                )
            )
            logger.info("Combined all attack types into a single label")
        logger.info(f"Data loaded into DataFrame with {len(self.data_df)} entries")
        logger.info(f"Label distribution:\n{self.data_df['label'].value_counts()}")

        if self.config.max_data > 0:
            logger.info(f"Limiting dataset to {self.config.max_data} samples per class")
            self.data_df = self.data_df.groupby("label").head(self.config.max_data)
        logger.info(
            f"Final dataset size: {len(self.data_df)} entries after applying max_data limit"
        )
        labels = self.data_df["label"].unique().tolist()
        self.label_encoding = {label: idx for idx, label in enumerate(labels)}
        for label in self.label_encoding.keys():
            lbl = [0] * len(self.label_encoding)
            idx = self.label_encoding[label]
            lbl[idx] = 1
            self.label_encoding[label] = lbl

        train_df, test_df = train_test_split(
            self.data_df,
            train_size=self.train_ratio,
            stratify=self.data_df["label"],
            random_state=self.config.random_seed,
        )
        self.train_df = train_df.reset_index(drop=True)
        self.test_df = test_df.reset_index(drop=True)
        logger.info(
            f"Split data into {len(self.train_df)} training and {len(self.test_df)} testing samples"
        )
        train_dataset = DFDataSet(self.config)
        train_dataset.data_df = self.train_df
        train_dataset.data_type = DataType.TRAIN
        train_dataset.label_encoding = self.label_encoding

        test_dataset = DFDataSet(self.config)
        test_dataset.data_df = self.test_df
        test_dataset.data_type = DataType.VALIDATION
        test_dataset.label_encoding = self.label_encoding
        logger.info(f"Created datasets: {train_dataset} and {test_dataset}")
        logger.info(
            f"{train_dataset.data_type}: {test_dataset.data_df.label.value_counts()}"
        )

        return train_dataset, test_dataset

    def __len__(self):
        if self.data_df is not None:
            return len(self.data_df)

    def __getitem__(self, idx):
        if self.data_df is not None:
            row = self.data_df.iloc[idx]
        image_path = Path(row["file_path"])
        label = row["label"]
        lbl_string = row.label

        if image_path.exists():
            img = cv2.imread(str(image_path))
            gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            h, w = gray_img.shape
            if h < self.config.num_pkts:
                # pad with zeros
                pad_height = self.config.num_pkts - h
                gray_img = cv2.copyMakeBorder(
                    gray_img, 0, pad_height, 0, 0, cv2.BORDER_CONSTANT, value=0
                )
            elif h > self.config.num_pkts:
                gray_img = gray_img[: self.config.num_pkts, :]
            if w < self.config.byte_length:
                # pad with zeros
                pad_width = self.config.byte_length - w
                gray_img = cv2.copyMakeBorder(
                    gray_img, 0, 0, 0, pad_width, cv2.BORDER_CONSTANT, value=0
                )
            elif w > self.config.byte_length:
                gray_img = gray_img[:, : self.config.byte_length]
            label = self.label_encoding[label] if self.label_encoding else label
            return gray_img, label, lbl_string

        raise IndexError("DataFrame is empty or not loaded.")

    def __repr__(self):
        return f"DataSet(data_root={self.data_root}, train_ratio={self.train_ratio})"


class TorchImageDataset(TorchDataset):
    def __init__(self, dataset: DFDataSet):
        self.dataset = dataset
        self.config = dataset.config
        self.data = dataset.data_df
        self.label_encoding = dataset.label_encoding
        self.data_type = dataset.data_type
        self.num_classes = len(self.label_encoding)
        # Get class counts
        class_counts = self.data["label"].value_counts().to_dict()

        # Compute inverse frequency weights
        class_weights = {label: 1.0 / count for label, count in class_counts.items()}

        # Order weights according to label encoding
        weights_list = [class_weights[label] for label in self.label_encoding.keys()]

        # Convert to tensor (no need to normalize)
        self.class_weights = torch.tensor(weights_list, dtype=torch.float32)
        self.class_weights = (
            self.class_weights * len(self.class_weights) / self.class_weights.sum()
        )
        self.label_counts = class_counts

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label, _ = self.dataset[idx]
        tensor = torch.from_numpy(image).float() / 255
        tensor = tensor.unsqueeze(0)
        label_tensor = torch.tensor(label, dtype=torch.float)
        return tensor, label_tensor

    def __repr__(self):
        return f"TorchImageDataset(dataset={self.dataset})"

In [ ]:
dfs = DFDataSet(PCAPImageDataConfig())
train_ds, test_ds = dfs.load_data()

In [ ]:
# plot some images
import matplotlib.pyplot as plt

rows = 3
cols = 3
fig, axes = plt.subplots(rows, cols, figsize=(12, 12))
for i in range(rows * cols):
    ax = axes[i // cols, i % cols]
    img, _, label = train_ds[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Train CNN

In [68]:
import torch.nn as nn


class SimpleCNN(nn.Module):
    def __init__(self, in_channel=1, num_classes=9):
        super(SimpleCNN, self).__init__()
        # model with global average pooling
        self.model = nn.Sequential(
            nn.Conv2d(in_channel, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.model(x)
        prob = nn.Softmax(dim=1)
        return x, prob(x)


from ids_expt.models.trainer import NNTrainer, NNTrainerConfig

trainer = NNTrainer(
    config=NNTrainerConfig(
        result_dir=Path(r"E:\MSc Works\IDS\results"),
        expt_name="pcap_image_classification",
        run_name="multi_classification",
        epochs=10,
        batch_size=8,
        learning_rate=0.001,
        device="cuda" if torch.cuda.is_available() else "cpu",
    ),
    model=SimpleCNN(
        in_channel=1,
        num_classes=len(train_ds.label_encoding),
    ),
    train_dataset=TorchImageDataset(train_ds),
    val_dataset=TorchImageDataset(test_ds),
)

In [ ]:
# logger.stop()
trainer.train()

## Found Sessions vs Labelled Sessions

In [ ]:
from pathlib import Path

stats_files = [f for f in Path(r"E:\MSc Works\IDS\output_updated").rglob("*.json")]
stat_df = pd.DataFrame()

for file in stats_files:
    with open(file, "r") as f:
        stats = json.load(f)
    stats["file"] = file.name
    stat_df = pd.concat([stat_df, pd.DataFrame([stats])], ignore_index=True)
stat_df

## Find Approx num PKTs and Byte length 

In [ ]:
self = dfs
self.all_sessions = list(self.data_root.rglob("*.pkl"))
logger.info(f"Found {len(self.all_sessions)} session files in {self.data_root}")
stats = []
session_index = 0
for session_file in tqdm(self.all_sessions):
    # session_file = self.all_sessions[43]
    with open(session_file, "rb") as f:
        try:
            sessions: list[Session] = pickle.load(f)
        except Exception as e:
            logger.error(f"Error reading {session_file}: {e}")
            continue

    for sidx, session in enumerate(sessions):
        session_pkts = session.packets
        num_pkts = len(session_pkts)
        label = session.label
        duration = session.duration
        pkt_lens = []
        for pkt in session_pkts:
            pkt_raw = raw(pkt)
            pkt_lens.append(len(pkt_raw))
        min_len = min(pkt_lens) if pkt_lens else 0
        max_len = max(pkt_lens) if pkt_lens else 0
        stats.append(
            {
                "session_file": session_file.name,
                "session_index": session_index,
                "num_packets": num_pkts,
                "label": label,
                "duration": duration,
                "min_packet_length": min_len,
                "max_packet_length": max_len,
            }
        )
        session_index += 1
self.stats_df = pd.DataFrame(stats)

In [ ]:
valid_stats = (
    self.stats_df.query("min_packet_length>0")
    .groupby("label")
    .agg(
        {
            "num_packets": ["mean", "min", "max"],
            "duration": ["mean", "min", "max"],
            "min_packet_length": ["mean", "min", "max"],
            "max_packet_length": ["mean", "min", "max"],
        }
    )
    .reset_index()
    .rename(columns={"label": "attack_type"})
)
valid_stats

In [ ]:
valid_stats.num_packets.mean().mean()

In [ ]:
(
    valid_stats.min_packet_length.mean().mean()
    + valid_stats.max_packet_length.mean().mean()
) / 2

In [ ]:
184 / 32, 234 / 32 # 6*32,8*32

In [35]:
self.stats_df.query("min_packet_length==0").label.value_counts().reset_index(
    name="count"
).rename(columns={"label": "attack_type"}).sort_values("count", ascending=False).to_csv(
    "cic_zero_min_packet_length.csv", index=False
)

In [36]:
valid_stats.to_csv("cic_valid_stats.csv", index=False)

In [ ]:
from cicflowmeter.flow_session import FlowSession
from cicflowmeter.sniffer import FlowSession

# sniffer = create_sniffer(
#     r"E:\MSc Works\IDS\data\Total PCAP Files\20200518_MITM_DOS_UOWM_DNP3_Dataset_Attacker_01.pcap",
#     None,
#     "csv",
#     "out.csv",
#     None,
#     True,
# )

In [3]:
from scapy.all import PcapReader

# packets = rdpcap(
#     r"E:\MSc Works\IDS\data\Total PCAP Files\20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_01.pcap"
# )

In [82]:
from cicflowmeter.flow_session import FlowSession

FlowSession.output_mode = "csv"
FlowSession.output = r"E:\MSc Works\IDS\notebooks\out.csv"
session = FlowSession()

i = 0
for packet in PcapReader(
    # r"E:\MSc Works\IDS\data\Total PCAP Files\20200516_DNP3_Enumerate_UOWM_DNP3_Dataset_Attacker_01.pcap"
    r"E:\MSc Works\IDS\data\DNP3 PCAP Files\20200516_DNP3_info_UOWM_DNP3_Dataset_Attacker_01.pcap"
    # r"E:\MSc Works\IDS\data\Total PCAP Files\20200516_DNP3_info_UOWM_DNP3_Dataset_Attacker_01.pcap"
):
    session.process(packet)

    # if i > 20000:
    #     break
    i += 1

In [83]:
# list(session.get_flows())[0].__dict__
# session.garbage_collect(list(session.get_flows())[-1].latest_timestamp)
[
    session.output_writer.write(flow.get_data(session.fields))
    for flow in session.get_flows()
]
list(session.get_flows())[0].packets[0][0].show2()
pass
# session.output_writer.writer.close()

In [95]:
list(session.get_flows())[0].packets[0][0].show2()

###[ Ethernet ]###
  dst       = be:0a:53:64:d9:b4
  src       = be:0a:53:35:94:71
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 1062
     id        = 15246
     flags     = DF
     frag      = 0
     ttl       = 64
     proto     = tcp
     chksum    = 0x77ef
     src       = 192.168.1.1
     dst       = 192.168.1.3
     \options   \
###[ TCP ]###
        sport     = 41020
        dport     = 20000
        seq       = 3239782653
        ack       = 1127006423
        dataofs   = 8
        reserved  = 0
        flags     = PA
        window    = 229
        chksum    = 0x6827
        urgptr    = 0
        options   = [('NOP', None), ('NOP', None), ('Timestamp', (7279472, 7277981))]
###[ Raw ]###
           load      = b'\x05d\x05\xc9\x00\x00\x00\x006L\x05d\x05\xc9\x01\x00\x00\x00\xde\x8e\x05d\x05\xc9\x02\x00\x00\x00\x9f\x84\x05d\x05\xc9\x03\x00\x00\x00wF\x05d\x05\xc9\x04\x00\x00\x00\x1d\x90\x05d\x05\xc9\x05\x00\x00\x00\xf5R\

In [84]:
import pandas as pd

df = pd.read_csv(
    # r"E:\MSc Works\IDS\data\CICFlowMeter\20200516_DNP3_Enumerate_UOWM_DNP3_Dataset_Attacker_01.pcap_FlowLABELED.csv"
    r"E:\MSc Works\IDS\data\Custom_DNP3_Parser\120_timeout\20200516_DNP3_info_UOWM_DNP3_Dataset_Attacker_01.pcapDNP3_FLOWLABELED.csv"
)
df.columns = df.columns.str.strip()
df["Timestamp"] = pd.to_datetime(df["date"])
# df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values(by="Timestamp").reset_index(drop=True)

C:\Users\Viper\AppData\Local\Temp\ipykernel_17348\3125294684.py:8: UserWarning: Parsing dates in  %d/%m/%Y  %H:%M:%S  format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(df["date"])


In [85]:
df.head(10)[[c for c in df.columns[:11] if c not in ["Flow ID", "Unnamed: 0"]]]

,flow ID,source IP,destination IP,source port,destination port,protocol,date,duration,TotalFwdPkts,TotalBwdPkts
0,192.168.1.1-192.168.1.3-40040-20000,192.168.1.1,192.168.1.3,40040,20000,6,16/05/2020 19:05:27,480.0,1,1
1,192.168.1.1-192.168.1.3-40042-20000,192.168.1.1,192.168.1.3,40042,20000,6,16/05/2020 19:05:54,179.0,1,1
2,192.168.1.1-192.168.1.3-40044-20000,192.168.1.1,192.168.1.3,40044,20000,6,16/05/2020 19:06:21,692.0,1,1
3,192.168.1.1-192.168.1.3-40046-20000,192.168.1.1,192.168.1.3,40046,20000,6,16/05/2020 19:06:50,269.0,1,1
4,192.168.1.1-192.168.1.3-40048-20000,192.168.1.1,192.168.1.3,40048,20000,6,16/05/2020 19:07:11,331.0,1,1
5,192.168.1.1-192.168.1.3-40050-20000,192.168.1.1,192.168.1.3,40050,20000,6,16/05/2020 19:07:34,272.0,1,1
6,192.168.1.1-192.168.1.3-40052-20000,192.168.1.1,192.168.1.3,40052,20000,6,16/05/2020 19:07:58,391.0,1,1
7,192.168.1.1-192.168.1.3-40054-20000,192.168.1.1,192.168.1.3,40054,20000,6,16/05/2020 19:08:23,273.0,1,1
8,192.168.1.1-192.168.1.3-40056-20000,192.168.1.1,192.168.1.3,40056,20000,6,16/05/2020 19:08:54,639.0,1,1
9,192.168.1.1-192.168.1.3-40058-20000,192.168.1.1,192.168.1.3,40058,20000,6,16/05/2020 19:09:18,330.0,1,1


In [86]:
odf = pd.read_csv(r"E:\MSc Works\IDS\notebooks\out.csv")
odf.timestamp = pd.to_datetime(odf.timestamp) + pd.Timedelta(hours=1)
odf = odf.sort_values(by="timestamp").reset_index(drop=True)
odf["flow_duration"] = odf.flow_duration.apply(
    lambda x: pd.Timedelta(seconds=float(x)).microseconds
)

# odf.query('timestamp>"2020-05-16 14:44:40"')
odf[
    [
        "src_ip",
        "dst_ip",
        "src_port",
        "dst_port",
        "protocol",
        "timestamp",
        "flow_duration",
        "tot_fwd_pkts",
        "tot_bwd_pkts",
    ]
]

,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration,tot_fwd_pkts,tot_bwd_pkts
0,192.168.1.3,192.168.1.1,40040,20000,6,2020-05-16 19:05:27,480,2,1
1,192.168.1.3,192.168.1.1,40042,20000,6,2020-05-16 19:05:54,179,2,1
2,192.168.1.3,192.168.1.1,40044,20000,6,2020-05-16 19:06:21,692,2,1
3,192.168.1.3,192.168.1.1,40046,20000,6,2020-05-16 19:06:50,269,2,1
4,192.168.1.3,192.168.1.1,40048,20000,6,2020-05-16 19:07:11,331,2,1
...,...,...,...,...,...,...,...,...,...
556,192.168.1.3,192.168.1.1,41152,20000,6,2020-05-16 23:03:04,312,2,1
557,192.168.1.3,192.168.1.1,41154,20000,6,2020-05-16 23:03:28,189,2,1
558,192.168.1.3,192.168.1.1,41156,20000,6,2020-05-16 23:03:54,244,2,1
559,192.168.1.3,192.168.1.1,41158,20000,6,2020-05-16 23:04:23,521,2,1


In [87]:
odf.columns
odf_cols = [
    "src_ip",
    "dst_ip",
    "src_port",
    "dst_port",
    "protocol",
    "timestamp",
    "flow_duration",
]
df_cols = [
    "source IP",
    "destination IP",
    "source port",
    "destination port",
    "protocol",
    "date",
    "duration",
]

In [88]:
ndf = df[df_cols].rename(
    columns={
        "source IP": "src_ip",
        "destination IP": "dst_ip",
        "source port": "src_port",
        "destination port": "dst_port",
        "protocol": "protocol",
        "date": "timestamp",
        "duration": "flow_duration",
    }
)

ndf[(ndf[odf_cols] != odf[odf_cols]).all(axis=1)]

,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration


In [216]:
from pydantic import BaseModel, Field
from pathlib import Path
from loguru import logger
from tqdm import tqdm
import pickle


class DNP3FlowDataConfig(BaseModel):
    pcap_dir: Path = Field(
        default=Path(r"E:\MSc Works\IDS\data\DNP3 PCAP Files"),
        description="Directory containing DNP3 PCAP files.",
    )
    csv_dir: Path = Field(
        default=Path(r"E:\MSc Works\IDS\data\Custom_DNP3_Parser\120_timeout"),
        description="Directory to save CSV files.",
    )
    output_dir: Path = Field(
        default=Path("output_dnp3sessions"),
        description="Directory to save output DNP3 session files.",
    )
    mapping_file: Path = Field(
        default=Path(r"E:\MSc Works\IDS\notebooks\dnp3_mapping.json"),
        description="Path to a mapping file for DNP3 flow data.",
    )


class DNP3FlowImageExtractor:
    def __init__(self, config: DNP3FlowDataConfig = DNP3FlowDataConfig()):
        self.config = config
        self.pcap_dir = config.pcap_dir
        self.csv_dir = config.csv_dir
        self.output_dir = config.output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)

        with open(config.mapping_file, "r") as f:
            self.mapping = json.load(f)

    def load_df(self, csv_file: Path):
        df = pd.read_csv(csv_file)
        df.columns = df.columns.str.strip()
        df["Timestamp"] = pd.to_datetime(df["date"])
        df = df.sort_values(by="Timestamp").reset_index(drop=True)
        return df

    def generate_features(self, pcap_file: Path):
        from scapy.all import PcapReader
        from cicflowmeter.flow_session import FlowSession

        FlowSession.output_mode = "csv"
        FlowSession.output = self.output_dir / f"{pcap_file.stem}_flow.csv"
        session = FlowSession()

        for packet in PcapReader(str(pcap_file)):
            session.process(packet)
        packets = []
        for flow in session.get_flows():
            session.output_writer.write(flow.get_data(session.fields))
            packets.extend(flow.packets)

        # flow_df = pd.read_csv(self.output_dir / f"{pcap_file.stem}_flow.csv")
        # flow_df.columns = flow_df.columns.str.strip()
        # flow_df["flow_duration"] = flow_df.flow_duration.apply(
        #     lambda x: pd.Timedelta(seconds=float(x)).microseconds
        # )
        # flow_df.to_csv(self.output_dir / f"{pcap_file.stem}_flow.csv", index=False)

        # Close the output writer to finalize the CSV file
        del session.output_writer
        return packets, self.output_dir / f"{pcap_file.stem}_flow.csv"

    def compare_flows(self, df: pd.DataFrame, flow_df: pd.DataFrame):
        flow_cols = [
            "src_ip",
            "dst_ip",
            "src_port",
            "dst_port",
            "protocol",
            "timestamp",
            "flow_duration",
        ]
        df_cols = df.columns.tolist()
        df = df[df_cols].rename(
            columns={
                "source IP": "src_ip",
                "destination IP": "dst_ip",
                "source port": "src_port",
                "destination port": "dst_port",
                "protocol": "protocol",
                "date": "timestamp",
                "duration": "flow_duration",
            }
        )
        df["original_index"] = df.index
        flow_df["flow_index"] = flow_df.index
        flow_df.columns = flow_df.columns.str.strip()

        # merge on flow columns
        merged_df = pd.merge(
            df[
                flow_cols
                + ["original_index", "Label", "TotalFwdPkts", "TotalBwdPkts", "Label"]
            ],
            flow_df[flow_cols + ["flow_index", "tot_fwd_pkts", "tot_bwd_pkts"]],
            on=flow_cols,
            how="inner",
        )
        return merged_df

    def run(self):
        pcap_files = list(self.pcap_dir.rglob("*.pcap"))
        pcap_files = sorted(pcap_files, key=lambda f: f.stat().st_size)
        logger.info(f"Found {len(pcap_files)} PCAP files in {self.pcap_dir}")
        for pcap_file in pcap_files:
            logger.info(f"Processing {pcap_file.name}")
            csv_fname = self.mapping.get(pcap_file.name)
            if not csv_fname:
                logger.warning(f"No mapping found for {pcap_file.name}, skipping.")
                continue
            csv_file = self.csv_dir / csv_fname
            logger.info(f"Corresponding CSV file: {csv_file.name}")
            if not csv_file.exists():
                logger.warning(f"CSV file {csv_file} does not exist, skipping.")
                continue
            # if "20200516_DNP3_info_UOWM_DNP3_Dataset_Attacker_01" not in pcap_file.name:
            #     continue
            df = self.load_df(csv_file)
            self.curr_df = df
            packets, flow_path = self.generate_features(pcap_file)
            # save packets
            with open(self.output_dir / f"{pcap_file.stem}_packets.pkl", "wb") as f:
                pickle.dump(packets, f)
            # flow_df = pd.read_csv(flow_path)
            # self.curr_flow_df = flow_df
            # self.curr_packets = packets
            # merged_df = self.compare_flows(df, flow_df)
            # logger.info(
            #     f"Found {len(merged_df)} matching rows in {pcap_file.stem} flows but {len(df)} rows in the original DataFrame."
            # )
            # pbar = tqdm(total=len(merged_df), desc=pcap_file.stem)
            # for i, row in merged_df.iterrows():
            #     pbar.update(1)
            #     flow_index = row["flow_index"]
            #     lbl_index = row["original_index"]
            #     session_packets = packets[flow_index]
            #     lbl_row = df.iloc[lbl_index]
            #     label = row["Label"]

            #     session = Session(
            #         label=label,
            #         packets=session_packets,
            #         duration=row["duration"],
            #         timestamp=row["Timestamp"],
            #         start_time=0,
            #         end_time=0,
            #         raw_bytes=[raw(pkt) for pkt in session_packets],
            #     )
            # break


config = DNP3FlowDataConfig()
extractor = DNP3FlowImageExtractor(config)
extractor.run()

2025-06-15 11:20:14.412 | INFO     | __main__:run:112 - Found 78 PCAP files in E:\MSc Works\IDS\data\DNP3 PCAP Files
2025-06-15 11:20:14.414 | INFO     | __main__:run:114 - Processing 20200518_Replay_UOWM_DNP3_Dataset_Attacker_03.pcap
2025-06-15 11:20:14.416 | INFO     | __main__:run:120 - Corresponding CSV file: 20200518_Replay_UOWM_DNP3_Dataset_Attacker_03.pcapDNP3_FLOWLABELED.csv
C:\Users\Viper\AppData\Local\Temp\ipykernel_17348\2563020329.py:41: UserWarning: Parsing dates in  %d/%m/%Y  %H:%M:%S  format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(df["date"])
2025-06-15 11:20:15.752 | INFO     | __main__:run:114 - Processing 20200518_Replay_UOWM_DNP3_Dataset_Attacker_02.pcap
2025-06-15 11:20:15.753 | INFO     | __main__:run:120 - Corresponding CSV file: 20200518_Replay_UOWM_DNP3_Dataset_Attacker_02.pcapDNP3_FLOWLABELED.csv
C:\Users\Viper\AppData\Local\Temp\ipykernel_17348\2563020

In [181]:
def compare_flows(self, df: pd.DataFrame, flow_df: pd.DataFrame):
    flow_cols = [
        "src_ip",
        "dst_ip",
        "src_port",
        "dst_port",
        "protocol",
        "timestamp",
        "flow_duration",
    ]
    df_cols = df.columns.tolist()
    df = df[df_cols].rename(
        columns={
            "source IP": "src_ip",
            "destination IP": "dst_ip",
            "source port": "src_port",
            "destination port": "dst_port",
            "protocol": "protocol",
            "date": "timestamp",
            "duration": "flow_duration",
        }
    )
    df["original_index"] = df.index
    flow_df["flow_index"] = flow_df.index
    flow_df.columns = flow_df.columns.str.strip()
    flow_df["flow_duration"] = flow_df.flow_duration.apply(
        lambda x: pd.Timedelta(seconds=float(x)).microseconds
    )
    flow_df.timestamp = pd.to_datetime(flow_df.timestamp) + pd.Timedelta(hours=1)
    df.timestamp = pd.to_datetime(df.timestamp)

    # merge on flow columns
    merged_df = pd.merge(
        df[
            flow_cols
            + ["original_index", "Label", "TotalFwdPkts", "TotalBwdPkts", "Label"]
        ],
        flow_df[flow_cols + ["flow_index", "tot_fwd_pkts", "tot_bwd_pkts"]],
        on=flow_cols,
        how="inner",
    )
    return merged_df


compare_flows(extractor, extractor.curr_df, extractor.curr_flow_df)
# extractor.curr_df

C:\Users\Viper\AppData\Local\Temp\ipykernel_17348\2753401493.py:30: UserWarning: Parsing dates in  %d/%m/%Y  %H:%M:%S  format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df.timestamp = pd.to_datetime(df.timestamp)


,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration,original_index,Label,TotalFwdPkts,TotalBwdPkts,Label,flow_index,tot_fwd_pkts,tot_bwd_pkts


In [212]:
df = extractor.curr_df.copy()
df = df.sort_values(by="date").reset_index(drop=True)
df.duration = df.duration.apply(lambda x: pd.Timedelta(microseconds=float(x)).seconds)
flow_df = extractor.curr_flow_df.copy()
flow_df.timestamp = pd.to_datetime(flow_df.timestamp) + pd.Timedelta(hours=1)
flow_df = flow_df.sort_values(by="timestamp").reset_index(drop=True)
# flow_df["flow_duration"] = flow_df.flow_duration.apply(
#     lambda x: pd.Timedelta(seconds=float(x)).microseconds
# )
df_cols = [
    "source IP",
    "destination IP",
    "source port",
    "destination port",
    "protocol",
    "date",
    "duration",
]

ndf = df[df_cols].rename(
    columns={
        "source IP": "src_ip",
        "destination IP": "dst_ip",
        "source port": "src_port",
        "destination port": "dst_port",
        "protocol": "protocol",
        "date": "timestamp",
        "duration": "flow_duration",
    }
)

ndf[(ndf[odf_cols] != odf[odf_cols]).all(axis=1)]

,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration


In [213]:
df.head(10)[[c for c in df.columns[:11] if c not in ["flow ID", "Unnamed: 0"]]]

,source IP,destination IP,source port,destination port,protocol,date,duration,TotalFwdPkts,TotalBwdPkts
0,192.168.1.1,192.168.1.3,40040,20000,6,16/05/2020 19:05:27,0,1,1
1,192.168.1.1,192.168.1.3,40042,20000,6,16/05/2020 19:05:54,0,1,1
2,192.168.1.1,192.168.1.3,40044,20000,6,16/05/2020 19:06:21,0,1,1
3,192.168.1.1,192.168.1.3,40046,20000,6,16/05/2020 19:06:50,0,1,1
4,192.168.1.1,192.168.1.3,40048,20000,6,16/05/2020 19:07:11,0,1,1
5,192.168.1.1,192.168.1.3,40050,20000,6,16/05/2020 19:07:34,0,1,1
6,192.168.1.1,192.168.1.3,40052,20000,6,16/05/2020 19:07:58,0,1,1
7,192.168.1.1,192.168.1.3,40054,20000,6,16/05/2020 19:08:23,0,1,1
8,192.168.1.1,192.168.1.3,40056,20000,6,16/05/2020 19:08:54,0,1,1
9,192.168.1.1,192.168.1.3,40058,20000,6,16/05/2020 19:09:18,0,1,1


In [164]:
config = DNP3FlowDataConfig()
pcap_files = [f.name for f in list(config.pcap_dir.rglob("*.pcap"))]
csv_files = [f.name for f in list(config.csv_dir.rglob("*.csv"))]
hard_mapping = {
    "20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_01.pcap": "20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_01.pcapDNP3_FLOWLABELED.csv",
    "20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_02.pcap": "20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_02.pcapDNP3_FLOWLABELED.csv",
    "20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_03.pcap": "20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_03.pcapDNP3_FLOWLABELED.csv",
    "20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Master.pcap": "20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Master.pcapDNP3_FLOWLABELED.csv",
    "20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_01.pcap": "20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_01.pcapDNP3_FLOWLABELED.csv",
    "20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_02.pcap": "20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_02.pcapDNP3_FLOWLABELED.csv",
    "20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_03.pcap": "20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_03.pcapDNP3_FLOWLABELED.csv",
    "20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_05.pcap": "20200508_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_05.pcapDNP3_FLOWLABELED.csv",
    "20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_06.pcap": "20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_06.pcapDNP3_FLOWLABELED.csv",
    "20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_01.pcap": "20200515_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_01.pcapDNP3_FLOWLABELED.csv",
    "20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_02.pcap": "20200515_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_02.pcapDNP3_FLOWLABELED.csv",
    "20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_03.pcap": "20200515_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_03.pcapDNP3_FLOWLABELED.csv",
    "20200515_DNP3_Warm_Restart_Attack_UOWM_DNP3_Dataset_Attacker_02.pcap": "20200515_Warm_Restart_Attack_UOWM_DNP3_Dataset_Attacker_02.pcapDNP3_FLOWLABELED.csv",
    "20200518_Replay_UOWM_DNP3_Dataset_Attacker_01.pcap": "20200518_Replay_UOWM_DNP3_Dataset_Attacker_01.pcapDNP3_FLOWLABELEDLABELED.csv",
}

mapping = {}
remaining_pcap = set(pcap_files)
remaining_csv = set(csv_files)
for pcap_file in pcap_files:
    csv_file = pcap_file + "DNP3_FLOWLABELED.csv"
    if csv_file in csv_files:
        mapping[pcap_file] = csv_file
        remaining_pcap.discard(pcap_file)
        remaining_csv.discard(csv_file)
    elif pcap_file in hard_mapping:
        mapping[pcap_file] = hard_mapping[pcap_file]
        remaining_pcap.discard(pcap_file)
        remaining_csv.discard(hard_mapping[pcap_file])

In [166]:
with open("dnp3_mapping.json", "w") as f:
    json.dump(mapping, f, indent=4)

In [159]:
hard_mapping

{'20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_01.pcap': '20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_01.pcapDNP3_FLOWLABELED.csv',
 '20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_02.pcap': '20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_02.pcapDNP3_FLOWLABELED.csv',
 '20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_03.pcap': '20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_03.pcapDNP3_FLOWLABELED.csv',
 '20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Master.pcap': '20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Master.pcapDNP3_FLOWLABELED.csv',
 '20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_01.pcap': '20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Slave_01.pcapDNP3_FLOWLABELED.csv',
 '20200514_DNP3_Disable_Unsolicited_Messages_Attack_UOWM